# Proyecto #

La idea de este documento es probar cosas y debugear

In [1]:
### Librerias ###
import pandas as pd
import numpy as np
import pandapower as pp
from pandapower.plotting import simple_plotly, pf_res_plotly
import math
import datetime

In [152]:
### Cargar red ###
project_net = pp.from_excel('../data/Project_net.xlsx')

### Cargar df ###
nuclear = pd.read_excel('../data/Nuclear.xlsx', names=["id","name","geoid","geoname","value[MW]","datetime"])
solar = pd.read_excel('../data/Solar.xlsx', names=["id","name","geoid","geoname","value[MW]","datetime"])
eolica = pd.read_excel('../data/Eolica.xlsx', names=["id","name","geoid","geoname","value[MW]","datetime"])
demanda = pd.read_excel('../data/DemandaReal.xlsx', names=["id","name","geoid","geoname","value[MW]","datetime"])

lista_df = [nuclear, solar, eolica, demanda]

for df in lista_df:
    df.drop(["id","name","geoid","geoname"], axis=1, inplace=True) # Estas columnas no tienen info
    df.drop(0, inplace=True) # La primera fila esta vacia
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True) # Casteo a datetime
    df["value[MW]"] = pd.to_numeric(df["value[MW]"]) # Casteo a numerico

In [157]:
### Normalise demand and generation profiles ### 

for df in lista_df:
    df["value[MW]"] = df["value[MW]"] / df["value[MW]"].max()

nuclear = nuclear.rename(columns={"value[MW]": "nuclear"})
solar   = solar.rename(columns={"value[MW]": "solar"})
eolica  = eolica.rename(columns={"value[MW]": "eolica"})
demanda = demanda.rename(columns={"value[MW]": "demanda"})

dataset = nuclear.merge(solar, on="datetime").merge(eolica, on="datetime").merge(demanda, on="datetime")
dataset = dataset.set_index("datetime")


In [160]:
dataset.head()

,nuclear,solar,eolica,demanda
datetime,,,,
2023-12-31 23:00:00+00:00,0.996664,0.001461,0.275834,0.575169
2023-12-31 23:00:00+00:00,0.996664,0.001461,0.275834,0.574915
2023-12-31 23:00:00+00:00,0.996664,0.001461,0.275834,0.578321
2023-12-31 23:00:00+00:00,0.996664,0.001461,0.275834,0.575169
2024-01-01 00:00:00+00:00,0.996734,0.001461,0.282552,0.557734


In [7]:
### Power Flow ###
pp.runpp(project_net)

In [14]:
project_net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.000000,0.000000,-111.148643,-88.990691
1,0.921620,-4.630386,0.000000,0.000000
2,0.858550,-7.031551,0.000000,0.000000
3,0.891422,-2.057614,0.000000,0.000000
4,1.000000,8.800996,-215.000000,-111.877628
5,0.792043,-14.673565,0.000000,0.000000
6,0.859300,-7.039271,0.000000,0.000000
7,NaN,NaN,0.000000,0.000000
8,NaN,NaN,0.000000,0.000000
9,NaN,NaN,0.000000,0.000000
